# Impact of Generative AI Synthetic Data on Text Classification with Limited Training Data

This notebook demonstrates the end-to-end experimental pipeline:
1. Loading and cleaning the SMS Spam Collection dataset.
2. Simulating a low-resource setting with restricted real training data (20%).
3. Augmenting training data with Generative AI synthetic examples (25%, 50%, 100%).
4. Evaluating text classification performance (Accuracy, Macro-F1, MCC, Training Time).
5. Comparing results across all experimental conditions.

In [1]:
import time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    classification_report
)

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Data Loading & Cleaning

We load the SMS spam dataset and apply robust data cleaning:
- Removing duplicate text entries to prevent data leakage and distortion.
- Normalizing unicode quotation marks, dashes, and currency symbols to standard ASCII.
- Sanitizing whitespace and validating label consistency (`ham` / `spam`).

In [2]:
import unicodedata
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFKD', text)
    replacements = {
        '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
        '\u2013': '-', '\u2014': '-', '\u2026': '...', '\u00a0': ' ',
        '\u0092': "'", '\u0093': '"', '\u0094': '"', '\u20b9': 'Rs '
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Load raw dataset and apply cleaning
raw_df = pd.read_csv('../data/sms_spam.csv')
print(f"Raw dataset shape: {raw_df.shape}")

# Apply cleaning and deduplication
df = raw_df.copy()
df.columns = df.columns.astype(str).str.strip().str.lower()
df = df[['text', 'label']].dropna()
df['text'] = df['text'].apply(clean_text)
df['label'] = df['label'].astype(str).str.strip().str.lower()
df = df[df['text'] != '']
df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)

print(f"Cleaned dataset shape: {df.shape} (Removed {len(raw_df) - len(df)} duplicate/invalid rows)")
print("\nCleaned class distribution:")
print(df['label'].value_counts())

display(df.head())

Raw dataset shape: (5157, 2)
Cleaned dataset shape: (5157, 2) (Removed 0 duplicate/invalid rows)

Cleaned class distribution:
label
ham     4516
spam     641
Name: count, dtype: int64


## 2. Dataset Splitting & Low-Resource Simulation

The dataset is divided into:
- **80% Training Set** (4,125 samples)
- **20% Test Set** (1,032 samples, 904 ham, 128 spam) - strictly held-out across all experiments.

To simulate a limited labelled data setting, only **20% of the training portion** is retained as real training data (825 samples: 722 ham, 103 spam).

In [3]:
X = df['text']
y = df['label']

# 80% train, 20% test stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Restrict available training data to 20% (low-resource setting)
X_real, _, y_real, _ = train_test_split(
    X_train, y_train, train_size=0.20, stratify=y_train, random_state=42
)

print(f"Total real training pool: {len(X_train)}")
print(f"Held-out test set samples: {len(X_test)}")
print(f"\nLimited real training samples (20% of train): {len(X_real)}")
print("Class distribution in limited real data:")
print(y_real.value_counts())

Total real training pool: 4125
Held-out test set samples: 1032

Limited real training samples (20% of train): 825
Class distribution in limited real data:
label
ham     722
spam    103
Name: count, dtype: int64


## 3. Model Pipeline

We use a standard, lightweight machine learning pipeline:
- **TF-IDF Vectorizer**: Converts message text into unigram and bigram features with English stop-words removed.
- **Logistic Regression**: Linear classifier (`max_iter=1000`, `random_state=42`).

In [4]:
def create_model(class_weight=None):
    return Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2))),
        ('classifier', LogisticRegression(max_iter=1000, class_weight=class_weight, random_state=42))
    ])

baseline_model = create_model()
print("Model pipeline architecture:")
print(baseline_model)

Model pipeline architecture:
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(ngram_range=(1, 2), stop_words='english')),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


## 4. Baseline Experiment: Real Data Only

We train the model solely on the limited real training data (825 samples) and evaluate on the 1,032 test samples.

In [5]:
start_time = time.perf_counter()
baseline_model.fit(X_real, y_real)
training_time = time.perf_counter() - start_time

y_pred = baseline_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
mcc = matthews_corrcoef(y_test, y_pred)

print("=== BASELINE RESULTS (REAL DATA ONLY) ===")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Macro-F1    : {macro_f1:.4f}")
print(f"MCC         : {mcc:.4f}")
print(f"Train Time  : {training_time:.4f} seconds")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

=== BASELINE RESULTS (REAL DATA ONLY) ===
Accuracy    : 0.8798
Macro-F1    : 0.4982
MCC         : 0.1658
Train Time  : 0.0264 seconds

Classification Report:
              precision    recall  f1-score   support

         ham       0.88      1.00      0.94       904
        spam       1.00      0.03      0.06       128

    accuracy                           0.88      1032
   macro avg       0.94      0.52      0.50      1032
weighted avg       0.89      0.88      0.83      1032


## 5. Synthetic Data Augmentation

We load the synthetic SMS dataset generated via ChatGPT.
The synthetic messages are cleaned and validated, then partitioned into nested subsets:
- **25% Synthetic**: 206 samples (180 ham, 26 spam)
- **50% Synthetic**: 413 samples (361 ham, 52 spam)
- **100% Synthetic**: 825 samples (722 ham, 103 spam)

In [6]:
synthetic_df = pd.read_csv('../data/synthetic_data.csv')
synthetic_df.columns = synthetic_df.columns.str.strip().str.lower()
synthetic_df['text'] = synthetic_df['text'].apply(clean_text)
synthetic_df = synthetic_df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"Cleaned synthetic dataset shape: {synthetic_df.shape}")
print("Synthetic class distribution:")
print(synthetic_df['label'].value_counts())

# Create nested subsets
ham_syn = synthetic_df[synthetic_df['label'] == 'ham'].sample(frac=1, random_state=42).reset_index(drop=True)
spam_syn = synthetic_df[synthetic_df['label'] == 'spam'].sample(frac=1, random_state=42).reset_index(drop=True)

n_real_ham = (y_real == 'ham').sum()
n_real_spam = (y_real == 'spam').sum()

synthetic_25 = pd.concat([ham_syn.iloc[:int(round(n_real_ham*0.25))], spam_syn.iloc[:int(round(n_real_spam*0.25))]], ignore_index=True)
synthetic_50 = pd.concat([ham_syn.iloc[:int(round(n_real_ham*0.50))], spam_syn.iloc[:int(round(n_real_spam*0.50))]], ignore_index=True)
synthetic_100 = pd.concat([ham_syn.iloc[:int(round(n_real_ham*1.0))], spam_syn.iloc[:int(round(n_real_spam*1.0))]], ignore_index=True)

print(f"\nSynthetic 25% : {len(synthetic_25)} samples -> {synthetic_25['label'].value_counts().to_dict()}")
print(f"Synthetic 50% : {len(synthetic_50)} samples -> {synthetic_50['label'].value_counts().to_dict()}")
print(f"Synthetic 100%: {len(synthetic_100)} samples -> {synthetic_100['label'].value_counts().to_dict()}")

Cleaned synthetic dataset shape: (888, 2)
Synthetic class distribution:
label
ham     768
spam    120
Name: count, dtype: int64

Synthetic 25% : 206 samples -> {'ham': 180, 'spam': 26}
Synthetic 50% : 413 samples -> {'ham': 361, 'spam': 52}
Synthetic 100%: 825 samples -> {'ham': 722, 'spam': 103}


## 6. Running All Experimental Conditions

We evaluate all four experimental conditions on the identical held-out test set:
1. **Real Only** (825 samples)
2. **Real + 25% Synthetic** (1,031 samples)
3. **Real + 50% Synthetic** (1,238 samples)
4. **Real + 100% Synthetic** (1,650 samples)

In [7]:
def run_experiment(name, X_tr, y_tr):
    model = create_model()
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    t_fit = time.perf_counter() - t0

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    mf1 = f1_score(y_test, preds, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_test, preds)

    return {
        'Experiment': name,
        'Real Samples': len(X_real),
        'Synthetic Samples': len(X_tr) - len(X_real),
        'Total Training Samples': len(X_tr),
        'Accuracy': acc,
        'Macro-F1': mf1,
        'MCC': mcc,
        'Training Time (sec)': t_fit
    }

exp_real = run_experiment('Real Only', X_real, y_real)
exp_25 = run_experiment('Real + 25% Synthetic', pd.concat([X_real, synthetic_25['text']], ignore_index=True), pd.concat([y_real, synthetic_25['label']], ignore_index=True))
exp_50 = run_experiment('Real + 50% Synthetic', pd.concat([X_real, synthetic_50['text']], ignore_index=True), pd.concat([y_real, synthetic_50['label']], ignore_index=True))
exp_100 = run_experiment('Real + 100% Synthetic', pd.concat([X_real, synthetic_100['text']], ignore_index=True), pd.concat([y_real, synthetic_100['label']], ignore_index=True))

results_df = pd.DataFrame([exp_real, exp_25, exp_50, exp_100])
display(results_df)

              Experiment  Real Samples  Synthetic Samples  Total Training Samples  Accuracy  Macro-F1       MCC  Training Time (sec)
0              Real Only           825                  0                     825    0.8798    0.4982    0.1658               0.0264
1   Real + 25% Synthetic           825                206                    1031    0.8828    0.5205    0.2196               0.0287
2   Real + 50% Synthetic           825                413                    1238    0.8847    0.5348    0.2493               0.0308
3  Real + 100% Synthetic           825                825                    1650    0.8905    0.5755    0.3227               0.0513


## 7. Save Results and Comparison Table

In [8]:
results_dir = Path('../results')
figures_dir = results_dir / 'figures'
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(results_dir / 'results.csv', index=False)

comparison_table = results_df[[
    'Experiment', 'Accuracy', 'Macro-F1', 'MCC', 'Training Time (sec)'
]].round(4)

comparison_table.to_csv(results_dir / 'comparison_table.csv', index=False)
print("Saved results.csv and comparison_table.csv")
display(comparison_table)

Saved results.csv and comparison_table.csv


## 8. Visualization of Results

We plot and save the four key comparison figures into `results/figures/`.

In [9]:
metrics_to_plot = [
    ('Accuracy', 'Accuracy Across Experimental Conditions', 'accuracy_comparison.png', '#2b5c8f', (0.85, 0.91)),
    ('Macro-F1', 'Macro-F1 Score Across Experimental Conditions', 'macro_f1_comparison.png', '#27ae60', (0.45, 0.62)),
    ('MCC', 'Matthews Correlation Coefficient (MCC) Comparison', 'mcc_comparison.png', '#e67e22', (0.12, 0.36)),
    ('Training Time (sec)', 'Training Time Across Experimental Conditions', 'training_time_comparison.png', '#8e44ad', None)
]

for col, title, filename, color, ylim in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    bars = ax.bar(comparison_table['Experiment'], comparison_table[col], color=color, width=0.55, edgecolor='black')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.set_ylabel(col, fontsize=11)
    ax.set_xlabel('Experimental Condition', fontsize=11)
    if ylim:
        ax.set_ylim(ylim)
    for bar in bars:
        yval = bar.get_height()
        label_txt = f"{yval:.4f}s" if 'Time' in col else f"{yval:.4f}"
        offset = 0.001 if 'Time' in col else (ylim[1] - ylim[0]) * 0.02
        ax.text(bar.get_x() + bar.get_width() / 2.0, yval + offset, label_txt, ha='center', va='bottom', fontsize=10, fontweight='bold')
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.savefig(figures_dir / filename)
    plt.show()
    plt.close()
    print(f"Generated and saved: {filename}")

Generated and saved: accuracy_comparison.png
Generated and saved: macro_f1_comparison.png
Generated and saved: mcc_comparison.png
Generated and saved: training_time_comparison.png
